# Unidad 1 — Extracción y procesamiento de texto
## Extracción de metadatos y sinopsis con Playwright + BeautifulSoup

**Sitio:** Lectulandia · **Categoría:** Thriller / Intriga
`https://ww3.lectulandia.co/genero/intriga/`

Este notebook construye un corpus de libros (**metadatos + sinopsis**) para usarlo
más adelante en tareas de procesamiento de texto y en la construcción de un
recomendador de libros.

**Reparto de responsabilidades entre herramientas**

| Herramienta | Rol en el proyecto |
|---|---|
| **Playwright** | Abre un Chromium real, recorre las páginas de la categoría y visita cada ficha. Ejecuta el JavaScript del sitio y devuelve el HTML ya renderizado. |
| **BeautifulSoup** | Analiza ese HTML y localiza los datos mediante selectores CSS. |
| **pandas** | Organiza, limpia, valida y exporta el dataset a `libros.csv`. |
| **Git** | Registra el trabajo en el repositorio personal de cada integrante. |

> **Importante:** el trabajo se limita a **metadatos y sinopsis públicas**.
> No se descargan libros ni archivos EPUB, PDF u otros contenidos.

**Cómo ejecutar:** `Entorno de ejecución → Ejecutar todo`. La celda de instalación
tarda 1–2 minutos; el scraping de 100 fichas ronda los 6–10 minutos por las pausas
entre visitas.

---
## 0. Instalación de dependencias

Se ejecuta **una sola vez por sesión** de Colab. `playwright install chromium`
descarga el navegador y `install-deps` instala las librerías del sistema que
Chromium necesita dentro de la máquina virtual de Colab.

In [ ]:
!pip install -q playwright beautifulsoup4 lxml pandas nest_asyncio
!playwright install chromium
!playwright install-deps chromium
print("Dependencias instaladas.")

---
## 1. Importaciones y configuración

Todos los parámetros que podrían cambiar (categoría, cantidad de libros, pausas,
rutas) están concentrados acá, de modo que el resto del código no necesite tocarse
si el grupo cambia de categoría.

`nest_asyncio.apply()` es imprescindible en Colab: el notebook ya tiene un *event
loop* de asyncio en marcha y, sin este parche, la API asincrónica de Playwright
falla con `This event loop is already running`.

In [ ]:
import asyncio
import csv
import random
import re
from datetime import date
from pathlib import Path
from urllib.parse import urljoin, urlparse

import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

nest_asyncio.apply()   # permite usar asyncio dentro del event loop de Colab

# ----------------------------- Configuración -----------------------------
BASE_URL         = "https://ww3.lectulandia.co"
CATEGORIA_NOMBRE = "Thriller / Intriga"
CATEGORIA_URL    = "https://ww3.lectulandia.co/genero/intriga/"

N_LIBROS_OBJETIVO    = 100      # la Parte 2 exige entre 50 y 100 fichas
MAX_PAGINAS          = 25       # tope de páginas de listado a recorrer
PAUSA_MIN, PAUSA_MAX = 1.5, 3.0 # pausa aleatoria entre visitas, en segundos
TIMEOUT_MS           = 45_000   # tiempo máximo de espera por página
HEADLESS             = True     # True = sin ventana (obligatorio en Colab)
VALOR_FALTANTE       = ""       # representación consistente de campos ausentes

# ------------------------------- Rutas -----------------------------------
DATA_DIR    = Path("data"); DATA_DIR.mkdir(exist_ok=True)
CSV_PARCIAL = DATA_DIR / "libros_parcial.csv"   # guardado incremental
CSV_FINAL   = DATA_DIR / "libros.csv"           # entregable

COLUMNAS = [
    "titulo", "autores", "generos", "serie", "sinopsis",
    "url_libro", "categoria_origen", "fecha_extraccion", "url_portada",
]
FECHA_EXTRACCION = date.today().isoformat()

print(f"Categoría : {CATEGORIA_NOMBRE}")
print(f"URL       : {CATEGORIA_URL}")
print(f"Objetivo  : {N_LIBROS_OBJETIVO} libros")
print(f"Fecha     : {FECHA_EXTRACCION}")

---
## 2. Funciones de limpieza de texto

Las sinopsis del sitio llegan con saltos de línea, sangrías y espacios duros
(`&nbsp;` → `\xa0`). Estas dos funciones normalizan **todo** lo que se escribe al
CSV, y cubren dos de los controles mínimos exigidos: eliminar espacios y saltos de
línea innecesarios, y representar los campos ausentes de manera consistente
(siempre la cadena vacía).

In [ ]:
ESPACIOS = re.compile(r"\s+")

def limpiar(texto):
    """Colapsa espacios y saltos de línea, y recorta los extremos.

    Devuelve VALOR_FALTANTE ("") cuando no hay texto útil, de modo que todos los
    campos ausentes del dataset se representen siempre de la misma manera.
    """
    if texto is None:
        return VALOR_FALTANTE
    texto = str(texto).replace("\xa0", " ")     # espacio duro -> espacio normal
    return ESPACIOS.sub(" ", texto).strip()

def unir(valores, sep="; "):
    """Une una lista de valores (autores, géneros) en un solo campo del CSV.

    Limpia cada valor, descarta los vacíos y elimina repetidos conservando el
    orden original (dict.fromkeys mantiene el orden de inserción).
    """
    limpios = [limpiar(v) for v in valores]
    limpios = [v for v in limpios if v]
    return sep.join(dict.fromkeys(limpios)) or VALOR_FALTANTE

# Prueba rápida
print(repr(limpiar("  Un  título\n  con   espacios\xa0raros ")))
print(repr(unir(["Intriga", " Novela ", "Intriga", ""])))
print(repr(limpiar(None)))

---
## 3. Lectura del listado de la categoría (BeautifulSoup)

De cada página de la categoría sólo necesitamos **las URL de las fichas**.
En lugar de atarnos a una clase CSS concreta del maquetado (que el sitio puede
cambiar), reconocemos las fichas por la **forma de su ruta**: `/book/<slug>/`
o `/libro/<slug>/`. Es un criterio mucho más estable.

Al mismo tiempo normalizamos cada enlace:

* `urljoin` convierte los enlaces relativos en absolutos;
* se descartan la *query string* y el fragmento (`?...`, `#...`), para que
  `/book/x/?ref=1` y `/book/x/` no cuenten como dos libros distintos;
* `dict.fromkeys` elimina repetidos manteniendo el orden.

In [ ]:
# Una ficha de libro tiene una ruta del tipo /book/<slug>/ (o /libro/<slug>/)
RE_FICHA = re.compile(r"^/(?:book|libro)/[^/]+/?$")

def extraer_urls_listado(html, base=BASE_URL):
    """Devuelve las URL absolutas y sin repetir de las fichas de una página."""
    soup = BeautifulSoup(html, "lxml")
    urls = []
    for a in soup.select("a[href]"):
        href = a["href"].split("?")[0].split("#")[0]   # sin query ni fragmento
        absoluta = urljoin(base, href)
        if RE_FICHA.match(urlparse(absoluta).path):
            urls.append(absoluta)
    return list(dict.fromkeys(urls))                   # sin duplicados, en orden

def url_pagina(n):
    """URL de la n-ésima página del listado de la categoría."""
    return CATEGORIA_URL if n == 1 else f"{CATEGORIA_URL.rstrip('/')}/page/{n}/"

print(url_pagina(1))
print(url_pagina(3))

---
## 4. Lectura de la ficha individual (BeautifulSoup)

Los selectores salen de inspeccionar el HTML del sitio con las herramientas de
desarrollo del navegador. La ficha organiza cada dato en un `<div>` con **id
propio**, y antepone un rótulo dentro de `<span class="tagTitle">`:

```html
<div id="title"><h1>TÍTULO</h1></div>
<div id="autor"    class="realign"><span class="tagTitle">Autor: </span>   <a class="dinSource" ...>AUTOR</a></div>
<div id="genero"   class="realign"><span class="tagTitle">Generos: </span> <a class="dinSource" ...>Intriga</a> <a ...>Novela</a></div>
<div id="sinopsis" class="realign"><span>Sinopsis</span> <p>…</p></div>
```

La función `_bloque()` resuelve el problema común a todos esos campos: **quitar el
rótulo** (`Autor:`, `Generos:`) para que no termine dentro del dato. Como cada
`id` aparece una sola vez en la página, hacer `decompose()` sobre ese `span` es
seguro.

Cada campo tiene además un **plan B**, para que una ficha con maquetado distinto
no se pierda: el título cae en el primer `<h1>` de la página, y la sinopsis en la
etiqueta `<meta name="description">`.

In [ ]:
def _bloque(soup, id_div):
    """Devuelve el <div id="..."> sin su rótulo (<span class="tagTitle">)."""
    div = soup.find("div", id=id_div)
    if div is None:
        return None
    for span in div.select("span.tagTitle"):
        span.decompose()          # elimina "Autor:", "Generos:", etc.
    return div

def _valores_enlazados(soup, id_div):
    """Texto de los <a> de un bloque (autores, géneros, serie), unidos con '; '."""
    div = _bloque(soup, id_div)
    if div is None:
        return VALOR_FALTANTE
    enlaces = [a.get_text() for a in div.select("a")]
    return unir(enlaces) if enlaces else limpiar(div.get_text())

def parsear_ficha(html, url):
    """Extrae los metadatos y la sinopsis de una ficha. Devuelve un dict-fila."""
    soup = BeautifulSoup(html, "lxml")

    # --- Título:  <div id="title"><h1>…</h1></div>  (plan B: primer <h1>) ---
    div_titulo = soup.find("div", id="title")
    titulo = limpiar(div_titulo.get_text()) if div_titulo else VALOR_FALTANTE
    if not titulo:
        h1 = soup.find("h1")
        titulo = limpiar(h1.get_text()) if h1 else VALOR_FALTANTE

    # --- Autores / géneros / serie: enlaces dentro de su <div id="…"> ---
    autores = _valores_enlazados(soup, "autor")
    generos = _valores_enlazados(soup, "genero")
    serie   = _valores_enlazados(soup, "serie")

    # --- Sinopsis: <div id="sinopsis"> con uno o varios <p> ---
    sinopsis = VALOR_FALTANTE
    div_sinopsis = _bloque(soup, "sinopsis")
    if div_sinopsis is not None:
        parrafos = [limpiar(p.get_text()) for p in div_sinopsis.find_all("p")]
        parrafos = [p for p in parrafos if p]
        sinopsis = " ".join(parrafos) if parrafos else limpiar(div_sinopsis.get_text())
        if sinopsis.lower().startswith("sinopsis"):      # rótulo suelto
            sinopsis = limpiar(sinopsis[len("sinopsis"):])
    if not sinopsis:                                     # plan B
        meta = soup.find("meta", attrs={"name": "description"})
        sinopsis = limpiar(meta.get("content")) if meta else VALOR_FALTANTE

    # --- Portada (campo opcional): sólo la URL de la imagen ---
    url_portada = VALOR_FALTANTE
    img = soup.select_one("div#cover img, .thumbnail img, article img")
    if img and img.get("src"):
        url_portada = urljoin(BASE_URL, img["src"])

    return {
        "titulo":           titulo,
        "autores":          autores,
        "generos":          generos,
        "serie":            serie,
        "sinopsis":         sinopsis,
        "url_libro":        url,
        "categoria_origen": CATEGORIA_NOMBRE,
        "fecha_extraccion": FECHA_EXTRACCION,
        "url_portada":      url_portada,
    }

# Prueba del parser con un HTML de ejemplo, sin salir a la red
HTML_DEMO = """
<html><body>
  <div id="title"><h1>  El   reclutador  </h1></div>
  <div id="autor" class="realign"><span class="tagTitle">Autor: </span>
      <a class="dinSource" href="/autor/david-mccloskey/">David McCloskey</a></div>
  <div id="genero" class="realign"><span class="tagTitle">Generos: </span>
      <a class="dinSource" href="/genero/intriga/">Intriga</a> &nbsp;
      <a class="dinSource" href="/genero/novela/">Novela</a></div>
  <div id="sinopsis" class="realign"><span>Sinopsis</span>
      <p>Primer párrafo de la
         sinopsis.</p><p>Segundo párrafo.</p></div>
</body></html>
"""
for k, v in parsear_ficha(HTML_DEMO, "https://ww3.lectulandia.co/book/demo/").items():
    print(f"{k:18}: {v}")

---
## 5. Navegación con Playwright y guardado incremental

Tres piezas:

* **`obtener_html()`** — navega a una URL y devuelve el HTML renderizado.
  Si algo falla (timeout, corte de red, error del servidor) **reintenta hasta 3
  veces con espera creciente** y, si aun así no lo logra, devuelve `None`. Así un
  libro problemático no interrumpe la corrida completa.
* **`guardar_fila()`** — escribe cada libro en el CSV parcial **apenas se
  obtiene**. Si Colab se desconecta a mitad de camino, no se pierde el trabajo.
* **`urls_ya_guardadas()`** — relee ese CSV al arrancar, de modo que una segunda
  ejecución **retome donde quedó** y no vuelva a visitar lo ya extraído.

In [ ]:
async def obtener_html(page, url, intentos=3):
    """Navega a `url` y devuelve su HTML renderizado, o None si falla.

    `wait_until="domcontentloaded"` no espera imágenes ni publicidad: basta con
    que el DOM esté armado. La espera extra da margen al JS del sitio.
    """
    for intento in range(1, intentos + 1):
        try:
            await page.goto(url, timeout=TIMEOUT_MS, wait_until="domcontentloaded")
            await page.wait_for_timeout(800)
            return await page.content()
        except Exception as e:
            print(f"      ! intento {intento}/{intentos} falló ({type(e).__name__}) en {url}")
            await asyncio.sleep(2 * intento)          # espera creciente
    return None

def urls_ya_guardadas(ruta=CSV_PARCIAL):
    """URL ya extraídas en corridas anteriores (permite reanudar sin duplicar)."""
    if not ruta.exists():
        return set()
    try:
        return set(pd.read_csv(ruta, dtype=str)["url_libro"].dropna())
    except Exception:
        return set()

def guardar_fila(fila, ruta=CSV_PARCIAL):
    """Agrega una fila al CSV parcial (guardado incremental)."""
    es_nuevo = not ruta.exists()
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        if es_nuevo:
            escritor.writeheader()
        escritor.writerow(fila)

print("Funciones de navegación y persistencia listas.")

---
## 6. Orquestador principal

Implementa, en orden, la estrategia de extracción declarada en la Parte 1:

1. abre la página de la categoría con Playwright;
2. recorre las páginas necesarias hasta llegar al objetivo;
3. obtiene el HTML con Playwright;
4. lo analiza con BeautifulSoup;
5. extrae las URL de las fichas;
6. visita cada ficha con Playwright (con **pausa aleatoria** previa);
7. extrae metadatos y sinopsis con BeautifulSoup;
8. limpia y valida (descarta fichas sin título);
9. evita duplicados mediante el conjunto `vistos`;
10. guarda cada registro en el CSV.

La pausa entre visitas es aleatoria en vez de fija: reduce la carga sobre el
servidor y hace el tráfico menos artificial.

In [ ]:
async def scrapear(objetivo=N_LIBROS_OBJETIVO, max_paginas=MAX_PAGINAS):
    """Recorre la categoría y extrae fichas hasta alcanzar `objetivo` libros."""
    vistos  = urls_ya_guardadas()          # evita duplicados y permite reanudar
    nuevos  = 0
    errores = 0
    print(f"Registros previos: {len(vistos)} · objetivo: {objetivo}\n")

    async with async_playwright() as p:
        navegador = await p.chromium.launch(headless=HEADLESS)
        contexto  = await navegador.new_context(
            user_agent=("Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"),
            locale="es-AR",
            viewport={"width": 1280, "height": 800},
        )
        pagina_web = await contexto.new_page()

        n_pagina = 1
        while len(vistos) < objetivo and n_pagina <= max_paginas:
            listado = url_pagina(n_pagina)
            print(f"[Listado] página {n_pagina}: {listado}")

            html_listado = await obtener_html(pagina_web, listado)
            if html_listado is None:
                errores += 1
                n_pagina += 1
                continue

            fichas = extraer_urls_listado(html_listado)
            print(f"          {len(fichas)} fichas encontradas")
            if not fichas:
                print("          Sin fichas: se detiene el recorrido de listados.")
                break

            for url in fichas:
                if len(vistos) >= objetivo:
                    break
                if url in vistos:                      # ya extraído: no duplicar
                    continue

                await asyncio.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))  # pausa

                html_ficha = await obtener_html(pagina_web, url)
                if html_ficha is None:
                    errores += 1
                    continue

                try:
                    fila = parsear_ficha(html_ficha, url)
                except Exception as e:
                    print(f"      ! error al analizar {url}: {type(e).__name__}: {e}")
                    errores += 1
                    continue

                if not fila["titulo"]:                 # validación mínima
                    print(f"      ! ficha sin título, se descarta: {url}")
                    errores += 1
                    continue

                guardar_fila(fila)                     # guardado incremental
                vistos.add(url)
                nuevos += 1
                print(f"      [{len(vistos):3d}/{objetivo}] {fila['titulo'][:60]}")

            n_pagina += 1

        await contexto.close()
        await navegador.close()

    print(f"\nFin. Nuevos: {nuevos} · Total acumulado: {len(vistos)} · Errores tolerados: {errores}")
    return nuevos

print("Orquestador definido.")

---
## 7. Ejecución del scraping

Colab admite `await` directamente en una celda. Si la sesión se corta, basta con
volver a ejecutar esta celda: el proceso **retoma** a partir de lo ya guardado en
`data/libros_parcial.csv`.

In [ ]:
await scrapear()

---
## 8. Limpieza, deduplicación y armado del dataset final (pandas)

Aunque cada fila ya se limpió al escribirla, se repasa el dataset completo:

* se normaliza otra vez el texto de todas las columnas;
* se descartan filas sin título o sin URL válida;
* se eliminan duplicados por `url_libro` (el control exigido) y, además, por
  la combinación `titulo` + `autores`, que detecta ediciones repetidas del mismo
  libro publicadas bajo distintas URL;
* se recorta al objetivo y se reordenan las columnas.

In [ ]:
df = pd.read_csv(CSV_PARCIAL, dtype=str)
print(f"Filas leídas del CSV parcial: {len(df)}")

# Asegura que estén todas las columnas y en el orden acordado
for col in COLUMNAS:
    if col not in df.columns:
        df[col] = VALOR_FALTANTE
df = df[COLUMNAS].fillna(VALOR_FALTANTE)

# Limpieza final de todas las celdas
for col in COLUMNAS:
    df[col] = df[col].map(limpiar)

antes = len(df)
df = df[df["titulo"] != ""]                                   # exige título
df = df[df["url_libro"].str.startswith("http")]               # exige URL válida
df = df.drop_duplicates(subset="url_libro", keep="first")     # control exigido
df = df.drop_duplicates(subset=["titulo", "autores"], keep="first")
df = df.head(N_LIBROS_OBJETIVO).reset_index(drop=True)

print(f"Filas descartadas por vacías o duplicadas: {antes - len(df)}")
print(f"Dataset final: {len(df)} libros")
df.head(5)

---
## 9. Controles mínimos

Verificación automática de los siete puntos exigidos por la consigna.
Si alguno falla, la celda lo informa y detiene la ejecución, para poder corregirlo
antes de entregar.

In [ ]:
def controles_minimos(df):
    sin_espacios_raros = not df.map(
        lambda v: isinstance(v, str) and (v != v.strip() or "\n" in v or "  " in v)
    ).to_numpy().any()

    controles = {
        "Sin duplicados por url_libro":              df["url_libro"].duplicated().sum() == 0,
        "Todos los registros tienen título":         (df["titulo"].str.len() > 0).all(),
        "Todas las URL son válidas":                 df["url_libro"].str.match(r"^https?://").all(),
        "La mayoría tiene sinopsis (>80%)":          (df["sinopsis"].str.len() > 0).mean() > 0.80,
        "Sin espacios ni saltos de línea sobrantes": sin_espacios_raros,
        "Campos ausentes representados igual":       df.isna().sum().sum() == 0,
        "Cantidad entre 50 y 100 libros":            50 <= len(df) <= 100,
    }
    ancho = max(len(k) for k in controles)
    for nombre, ok in controles.items():
        print(f"{'  OK ' if ok else 'FALLA'}  {nombre:<{ancho}}")
    print()
    print(f"Libros            : {len(df)}")
    print(f"Con sinopsis      : {(df['sinopsis'].str.len() > 0).sum()} "
          f"({(df['sinopsis'].str.len() > 0).mean():.1%})")
    print(f"Con serie         : {(df['serie'].str.len() > 0).sum()}")
    print(f"Autores distintos : {df['autores'].nunique()}")
    return all(controles.values())

assert controles_minimos(df), "Hay controles mínimos que no se cumplen: revisar antes de entregar."
print("\nTodos los controles mínimos se cumplen.")

---
## 10. Exportación de `libros.csv`

Se guarda en `data/libros.csv` (la ruta que pide la consigna) y se dispara la
descarga al equipo, para poder subirlo al repositorio.

In [ ]:
df.to_csv(CSV_FINAL, index=False, encoding="utf-8")
print(f"Guardado: {CSV_FINAL.resolve()}  "
      f"({len(df)} filas, {CSV_FINAL.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files
    files.download(str(CSV_FINAL))
except Exception:
    print("(Descarga automática no disponible fuera de Colab: el archivo ya está en disco.)")

---
## 11. Exploración rápida del corpus

Un vistazo al dataset obtenido, útil para el informe y como punto de partida para
las actividades de procesamiento de texto de las próximas unidades.

In [ ]:
df["n_palabras_sinopsis"] = df["sinopsis"].str.split().str.len()

print("=== Longitud de las sinopsis (en palabras) ===")
print(df["n_palabras_sinopsis"].describe().round(1).to_string())

print("\n=== Autores más frecuentes ===")
print(df["autores"].value_counts().head(10).to_string())

print("\n=== Géneros más frecuentes ===")
generos = df["generos"].str.split("; ").explode().str.strip()
print(generos[generos != ""].value_counts().head(10).to_string())

print("\n=== Completitud por campo (%) ===")
print(((df[COLUMNAS] != "").mean() * 100).round(1).to_string())

ax = df["n_palabras_sinopsis"].plot(
    kind="hist", bins=25,
    title="Distribución de la longitud de las sinopsis",
    xlabel="palabras", ylabel="cantidad de libros",
)

---
## 12. Registro en Git

Cada integrante debe incorporar el trabajo a su repositorio personal, con esta
estructura:

```
README.md
src/
  scraper.py
data/
  libros.csv
docs/
  diseno_extraccion.md
```

Lo habitual es descargar `libros.csv` con la celda 10 y hacer el *commit* desde la
computadora:

```bash
git add README.md src/ data/libros.csv docs/
git commit -m "Unidad 1: corpus de 100 libros de la categoría Intriga"
git push
```

> **Nota:** no subir credenciales ni tokens dentro del notebook.